<a href="https://colab.research.google.com/github/sameerkarur/Data_science/blob/main/06_IITK_AIML_Advanced_Generative_AI/lms_upload_project1/hr_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Course-End Project 1: AI-Powered HR Assistant (Nestlé HR Policy)

**Objective:** Build a conversational RAG chatbot that answers HR policy questions from Nestlé's HR PDF using LangChain, OpenAI embeddings, ChromaDB, GPT-3.5 Turbo, and Gradio.

## Step 1: Install dependencies & set up OpenAI API

In [1]:
%pip install -q langchain langchain-community langchain-openai langchain-text-splitters chromadb pypdf gradio python-dotenv


[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: /Users/sameerkarur/Documents/Git/Data_science/projects/genai_course_end/.venv/bin/python3.14 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
from getpass import getpass
from pathlib import Path

SECRETS_PATH = Path.home() / "Documents" / "opencareerai-secrets.json"

if not os.environ.get("OPENAI_API_KEY") and SECRETS_PATH.exists():
    with open(SECRETS_PATH, encoding="utf-8") as f:
        secrets = json.load(f)
    os.environ["OPENAI_API_KEY"] = secrets["ai_apis"]["openai_api_key"]
    print("Loaded OPENAI_API_KEY from opencareerai-secrets.json")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

PDF_PATH = "Dataset/the_nestle_hr_policy_pdf_2012.pdf"
CHROMA_DIR = "chroma_hr_db"

Loaded OPENAI_API_KEY from opencareerai-secrets.json


## Step 2: Load Nestlé HR policy PDF and split into chunks

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()
print(f"Loaded {len(documents)} pages from HR policy PDF")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} text chunks")
print("Sample chunk:", chunks[0].page_content[:300], "...")

/var/folders/hp/ftdqsbms3c16tllqdp4hz1dm0000gn/T/ipykernel_47298/3282298604.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 8 pages from HR policy PDF
Created 20 text chunks
Sample chunk: Policy
Mandatory
September  2012
The Nestlé  
Human Resources Policy ...


## Step 3: Create vector embeddings with OpenAI & store in ChromaDB

In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
)

if hasattr(vectorstore, "persist"):
    vectorstore.persist()

print("Vector store created and persisted.")

Vector store created and persisted.


/var/folders/hp/ftdqsbms3c16tllqdp4hz1dm0000gn/T/ipykernel_47298/3406136696.py:13: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


## Step 4: Build QA chain with prompt template (GPT-3.5 Turbo)

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

prompt = ChatPromptTemplate.from_template(
    """You are Nestlé's HR policy assistant. Answer ONLY using the provided context.
If the answer is not in the context, say you don't have that information in the HR policy document.
Be concise, professional, and accurate.

Context:
{context}

Question: {input}

Answer:"""
)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


qa_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [6]:
def ask_hr(question: str) -> str:
    return qa_chain.invoke(question)


# Quick test
sample_q = "What is Nestlé's policy on annual leave?"
print("Q:", sample_q)
print("A:", ask_hr(sample_q))

Q: What is Nestlé's policy on annual leave?


A: I don't have that information in the HR policy document.


## Step 5: Gradio chatbot interface

In [7]:
import gradio as gr


def chatbot_response(message, history):
    answer = ask_hr(message)
    return answer


demo = gr.ChatInterface(
    fn=chatbot_response,
    title="Nestlé HR Policy Assistant",
    description=(
        "Ask questions about Nestlé HR policies. Answers are retrieved from the official HR policy PDF using RAG."
    ),
    examples=[
        "What is the recruitment policy?",
        "What are the working hours guidelines?",
        "What is the policy on employee benefits?",
    ],
)

# Test Q&A (for notebook execution)
for q in [
    "What is the recruitment policy?",
    "What are the working hours guidelines?",
]:
    print(f"Q: {q}")
    print(f"A: {ask_hr(q)}\n")

/Users/sameerkarur/Documents/Git/Data_science/projects/genai_course_end/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Q: What is the recruitment policy?


A: I don't have that information in the HR policy document.

Q: What are the working hours guidelines?


A: We provide flexible working conditions whenever possible to support a better balance of private and professional life.



### Launch Gradio UI (run locally for screenshots)

Uncomment and run the cell below to open the interactive chatbot in your browser.

In [8]:
# demo.launch(share=False)

## Conclusion

This notebook demonstrates the full workflow:
1. PDF loading with `PyPDFLoader`
2. Text chunking for retrieval
3. OpenAI embeddings + ChromaDB vector store
4. GPT-3.5 Turbo QA with a structured prompt template
5. Gradio UI for user interaction